In [5]:
# Load config.yaml and summarize DB settings
from pathlib import Path
import yaml

cfg_path = Path('config.yaml')
if not cfg_path.exists():
    print('config.yaml not found at project root')
else:
    cfg = yaml.safe_load(cfg_path.read_text())
    print('LLM provider/model:', cfg.get('llm',{}).get('provider'), cfg.get('llm',{}).get('model'))
    db = cfg.get('database', {})
    print('DB driver:', db.get('driver'))
    print('DB server:', db.get('server'))
    print('DB database:', db.get('database'))
    print('DB username:', db.get('username'))

config.yaml not found at project root


In [6]:
# List installed ODBC drivers
import pyodbc
print('pyodbc version:', pyodbc.version)
print('Available drivers:')
for d in pyodbc.drivers():
    print('-', d)

pyodbc version: 5.3.0
Available drivers:
- ODBC Driver 18 for SQL Server


In [7]:
# Basic TCP connectivity test to SQL Server host (port 1433)
import socket, yaml

cfg = yaml.safe_load(Path('config.yaml').read_text()) if Path('config.yaml').exists() else {}
server = (cfg.get('database', {}) or {}).get('server')
if not server:
    print('No server in config.yaml')
else:
    host = server
    # If instance notation host\INSTANCE, extract host

No server in config.yaml


In [10]:
# Attempt an ODBC connection using config.yaml settings
import pyodbc, yaml

cfg = yaml.safe_load(Path('config.yaml').read_text()) if Path('config.yaml').exists() else {}
db = cfg.get('database', {})
server = db.get('server')
database = db.get('database')
username = db.get('username')
password = db.get('password')
driver = db.get('driver') or 'ODBC Driver 18 for SQL Server'

if not server or not database or not username or not password:
    print('Missing DB settings (server/database/username/password) in config.yaml')
else:
    # Instance notation to host; ODBC connection supports host\instance but many Linux setups require explicit host:port
    # We'll try host\instance first, then fallback to host,1433 if needed
    conn_str = f'DRIVER={{{driver}}};SERVER={server};DATABASE={database};UID={username};PWD={password};TrustServerCertificate=YES;'
    print('Connecting with:', conn_str.replace(password, '***'))
    try:
        cn = pyodbc.connect(conn_str, timeout=5)
        print('✅ ODBC connect succeeded')
        cn.close()
    except Exception as e:
        print('❌ ODBC connect failed:', e)
        # Fallback: try host:1433 if instance notation present
        if '\' in server:
            host = server.split('\',1)[0]
            conn_str2 = f'DRIVER={{{driver}}};SERVER={host},1433;DATABASE={database};UID={username};PWD={password};TrustServerCertificate=YES;'
            print('Retry with host:port:', conn_str2.replace(password, '***'))
            try:
                cn = pyodbc.connect(conn_str2, timeout=5)
                print('✅ ODBC connect (host:port) succeeded')
                cn.close()
            except Exception as e2:
                print('❌ ODBC connect (host:port) failed:', e2)

SyntaxError: unterminated string literal (detected at line 26) (391393694.py, line 26)

In [11]:
# Improved ODBC connection test (find config in parent, env fallback)
import os
from pathlib import Path
import pyodbc, yaml

# Locate config.yaml (current dir, parent dir)
cfg_path = None
for candidate in [Path('config.yaml'), Path('../config.yaml')]:
    if candidate.exists():
        cfg_path = candidate
        break

cfg = {}
if cfg_path:
    print(f"Using config file: {cfg_path}")
    try:
        cfg = yaml.safe_load(cfg_path.read_text()) or {}
    except Exception as e:
        print(f"Failed to read config.yaml: {e}")
else:
    print("config.yaml not found in current or parent directory; using environment variables if present.")

db_cfg = cfg.get('database', {})
# Environment variable overrides
server = os.environ.get('DB_SERVER') or db_cfg.get('server')
database = os.environ.get('DB_NAME') or db_cfg.get('database')
username = os.environ.get('DB_USER') or db_cfg.get('username')
password = os.environ.get('DB_PASSWORD') or db_cfg.get('password')
driver = db_cfg.get('driver') or os.environ.get('ODBC_DRIVER') or 'ODBC Driver 18 for SQL Server'

if not server or not database or not username or not password:
    print('Missing DB settings (server/database/username/password). Set in config.yaml or env (DB_SERVER, DB_NAME, DB_USER, DB_PASSWORD).')
else:
    # Try host\\instance first, then fallback to host:1433 if needed
    conn_str = f"DRIVER={{{driver}}};SERVER={server};DATABASE={database};UID={username};PWD={password};TrustServerCertificate=YES;"
    print('Connecting with:', conn_str.replace(password, '***'))
    try:
        cn = pyodbc.connect(conn_str, timeout=5)
        print('✅ ODBC connect succeeded')
        cn.close()
    except Exception as e:
        print('❌ ODBC connect failed:', e)
        # Fallback: try host:1433 if instance notation present
        if '\\' in server:
            host = server.split('\\', 1)[0]
            conn_str2 = f"DRIVER={{{driver}}};SERVER={host},1433;DATABASE={database};UID={username};PWD={password};TrustServerCertificate=YES;"
            print('Retry with host:port:', conn_str2.replace(password, '***'))
            try:
                cn = pyodbc.connect(conn_str2, timeout=5)
                print('✅ ODBC connect (host:port) succeeded')
                cn.close()
            except Exception as e2:
                print('❌ ODBC connect (host:port) failed:', e2)


Using config file: ../config.yaml
Connecting with: DRIVER={ODBC Driver 18 for SQL Server};SERVER=localhost\\SQLEXPRESS;DATABASE=kam;UID=your_username;PWD=***;TrustServerCertificate=YES;
❌ ODBC connect failed: ('HYT00', '[HYT00] [Microsoft][ODBC Driver 18 for SQL Server]Login timeout expired (0) (SQLDriverConnect)')
Retry with host:port: DRIVER={ODBC Driver 18 for SQL Server};SERVER=localhost,1433;DATABASE=kam;UID=your_username;PWD=***;TrustServerCertificate=YES;
❌ ODBC connect (host:port) failed: ('HYT00', '[HYT00] [Microsoft][ODBC Driver 18 for SQL Server]Login timeout expired (0) (SQLDriverConnect)')


In [ ]:
# Inspect latest diagnostics bundle manifest if present
from pathlib import Path
import zipfile, json

zip_files = sorted((Path('logs/diagnostics').glob('diagnostics_bundle_*.zip')), reverse=True)
if not zip_files:
    print('No diagnostics bundle zips found under logs/diagnostics')
else:
    zp = zip_files[0]
    print('Using bundle:', zp)
    with zipfile.ZipFile(zp) as zf:
        if 'manifest.json' in zf.namelist():
            manifest = json.loads(zf.read('manifest.json').decode('utf-8'))
            print('Manifest paths (first 10):')
            for p in manifest.get('paths', [])[:10]:
                print('-', p)
        else:
            print('manifest.json not found in bundle')

## Remediation Checklist (SQL Server on Linux dev containers)
- Ensure SQL Server is reachable from the container: host and port 1433 open.
- If using instance notation (\SQLEXPRESS), prefer explicit host:port in Linux.
- Verify ODBC Driver 18 is installed; driver name must match `pyodbc.drivers()`.
- Enable TCP/IP in SQL Server and confirm `ip:1433` is listening.
- Confirm credentials and database exist; test with `sqlcmd` or `pyodbc`.
- For local dev, consider Docker `mcr.microsoft.com/mssql/server` and update `config.yaml` to the container's host/port.

In [ ]:
# Quick environment variable check for LLM providers
import os
print('OPENAI_API_KEY present:', bool(os.environ.get('OPENAI_API_KEY')))
print('GROQ_API_KEY present:', bool(os.environ.get('GROQ_API_KEY')))